# Toy particle system

2D particle system of $N$ particles with exactlty $N/2$ species and two particles per species.

In [ ]:
import diffrax as dfx
import distrax as dsx
import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib
import matplotlib.pyplot as plt
import optax

from superiorflows import Flow
from superiorflows.train import (
    EnergyBasedLoss,
    KullbackLeiblerLoss,
    LoggerCallback,
    MaximumLikelihoodLoss,
    ProgressBarCallback,
    CheckpointCallback,
    Trainer,
)

class ToyParticles(eqx.Module):
    positions: jnp.ndarray
    species: jnp.ndarray
    box: jnp.ndarray

    @property
    def N(self):
        return self.positions.shape[-2]


/Users/Leonardo/Documents/Postdoc/Projects/superiorflows/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
n_species = 3
alphas = jnp.ones(n_species)
betas = jnp.ones(n_species)
L = 2.0

assert alphas.shape == betas.shape
N = 2 * len(alphas)
prior_dist = dsx.Uniform(low=jnp.zeros(2), high=L*jnp.ones(2))
angle_dist = dsx.Uniform(low=jnp.zeros(2), high=2*jnp.pi*jnp.ones(2))
norm_dist = dsx.Transformed(
    distribution=dsx.Beta(
        alpha=alphas,
        beta=betas
    ),
    bijector=dsx.ScalarAffine(
        shift=jnp.zeros((n_species,)), 
        scale=(L / 2.0) * jnp.ones((n_species,))
    )
)